# Step 5 Part F: Evaluate v2 (turnover-penalty model) on TEST, compare against v1 and classical benchmarks

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import json
from scipy.stats import norm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BTC_TRANSACTION_COST_RATE = 0.0005
MAX_LEN = 24
MIN_LEN = 4
feature_names = ["moneyness", "ttm", "iv", "delta", "is_call"]
norm_stats = json.load(open("norm_stats.json"))

def bs_delta(S, K, T, sigma, option_type, r=0.0):
    valid = (T > 0) & (sigma > 0)
    d1 = np.where(valid, (np.log(S / K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(np.where(valid, T, 1))), 0)
    call_delta = norm.cdf(d1)
    put_delta = call_delta - 1.0
    is_call = (option_type == "call")
    return np.where(valid, np.where(is_call, call_delta, put_delta), 0.0)

def build_episode_arrays(df, max_len=MAX_LEN, min_len=MIN_LEN):
    features_list, spot_list, option_pnl_list, mask_list = [], [], [], []
    meta = []
    for (symbol, sample_date), group in df.groupby(["symbol", "sample_date"]):
        group = group.sort_values("hour_bucket").reset_index(drop=True)
        n = len(group)
        if n < min_len:
            continue
        n_use = min(n, max_len)
        group = group.iloc[:n_use]
        moneyness = (group["underlying_price"] / group["strike_price"]).values
        ttm = group["T_years"].values
        iv = group["iv_decimal"].clip(upper=3.0).values
        delta = group["bs_delta"].values
        is_call = (group["type"] == "call").astype(float).values
        feat = np.stack([moneyness, ttm, iv, delta, is_call], axis=1)
        spot = group["underlying_price"].values
        option_mid_usd = group["option_mid_usd"].values
        option_pnl = -np.diff(option_mid_usd, prepend=option_mid_usd[0])
        option_pnl[0] = 0.0
        pad_n = max_len - n_use
        if pad_n > 0:
            feat = np.vstack([feat, np.zeros((pad_n, feat.shape[1]))])
            spot = np.concatenate([spot, np.full(pad_n, spot[-1])])
            option_pnl = np.concatenate([option_pnl, np.zeros(pad_n)])
        mask = np.array([1.0] * n_use + [0.0] * pad_n)
        features_list.append(feat)
        spot_list.append(spot)
        option_pnl_list.append(option_pnl)
        mask_list.append(mask)
        meta.append({"symbol": symbol, "sample_date": sample_date, "n_steps": n_use,
                      "avg_moneyness": moneyness[:n_use].mean(), "avg_ttm_days": (ttm[:n_use] * 365).mean(),
                      "option_type": group["type"].iloc[0]})
    return (np.stack(features_list), np.stack(spot_list), np.stack(option_pnl_list),
            np.stack(mask_list), pd.DataFrame(meta))

def normalize_features(features, stats):
    normed = features.copy()
    for i, name in enumerate(feature_names):
        normed[:, :, i] = (features[:, :, i] - stats[name]["mean"]) / stats[name]["std"]
    return normed

test = pd.read_csv("btc_options_test.csv")
test["hour_bucket"] = pd.to_datetime(test["hour_bucket"])
test["sample_date"] = test["hour_bucket"].dt.date
test["T_years"] = test["time_to_maturity_days"] / 365
test["iv_decimal"] = test["mark_iv"] / 100
test["option_mid_usd"] = test["mid_price"] * test["underlying_price"]
test["bs_delta"] = bs_delta(test["underlying_price"].values, test["strike_price"].values,
                               test["T_years"].values, test["iv_decimal"].values, test["type"].values)

test_features, test_spots, test_option_pnls, test_masks, test_meta = build_episode_arrays(test)
test_features_normed = normalize_features(test_features, norm_stats)
print(f"Test: {test_features.shape[0]} episodes")

## Load v2 model and run on test

In [ ]:
class DeepHedgePolicy(nn.Module):
    def __init__(self, input_size=5, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        raw_position = self.head(lstm_out)
        return 1.5 * torch.tanh(raw_position.squeeze(-1))

model_v2 = DeepHedgePolicy().to(device)
model_v2.load_state_dict(torch.load("best_deep_hedge_model_v2.pt", map_location=device))
model_v2.eval()

test_features_t = torch.tensor(test_features_normed, dtype=torch.float32).to(device)
test_spots_t = torch.tensor(test_spots, dtype=torch.float32).to(device)
test_option_pnls_t = torch.tensor(test_option_pnls, dtype=torch.float32).to(device)
test_masks_t = torch.tensor(test_masks, dtype=torch.float32).to(device)

with torch.no_grad():
    positions = model_v2(test_features_t)
    batch_size, seq_len = positions.shape
    prev_position = torch.cat([torch.zeros(batch_size, 1, device=device), positions[:, :-1]], dim=1)
    trade = (positions - prev_position) * test_masks_t
    cost = trade.abs() * test_spots_t * (BTC_TRANSACTION_COST_RATE / 2)
    prev_spot = torch.cat([test_spots_t[:, :1], test_spots_t[:, :-1]], dim=1)
    hedge_pnl = prev_position * (test_spots_t - prev_spot)
    total_pnl_per_step = (test_option_pnls_t + hedge_pnl - cost) * test_masks_t
    terminal_pnl = total_pnl_per_step.sum(dim=1)
    turnover = trade.abs().sum(dim=1)
    total_cost = cost.sum(dim=1)
    n_trades = (trade.abs() > 1e-6).sum(dim=1)

print(f"v2 terminal P&L: mean={terminal_pnl.mean().item():.4f}, std={terminal_pnl.std().item():.4f}")
print(f"v2 total cost: mean={total_cost.mean().item():.4f}")
print(f"v2 turnover: mean={turnover.mean().item():.4f}")
print(f"v2 n_trades: mean={n_trades.float().mean().item():.4f}")

## Combine with existing results (v1 deep hedge + classical benchmarks) and compare all 5

In [ ]:
v2_results = test_meta.copy()
v2_results["strategy"] = "deep_hedge_v2"
v2_results["total_cost"] = total_cost.cpu().numpy()
v2_results["turnover"] = turnover.cpu().numpy()
v2_results["n_trades"] = n_trades.cpu().numpy()
v2_results["terminal_pnl"] = terminal_pnl.cpu().numpy()
v2_results["sample_date"] = pd.to_datetime(v2_results["sample_date"]).dt.date

existing = pd.read_csv("test_results_with_deep_hedge.csv")
existing["sample_date"] = pd.to_datetime(existing["sample_date"]).dt.date

combined = pd.concat([existing, v2_results], ignore_index=True)
combined.to_csv("test_results_all_strategies.csv", index=False)

def cvar_95(pnl_series):
    threshold = pnl_series.quantile(0.05)
    return pnl_series[pnl_series <= threshold].mean()

rows = []
for strategy, group in combined.groupby("strategy"):
    pnl = group["terminal_pnl"]
    rows.append({
        "strategy": strategy, "mean_pnl": pnl.mean(), "std_pnl": pnl.std(),
        "cvar_95": cvar_95(pnl), "mean_cost": group["total_cost"].mean(),
        "mean_turnover": group["turnover"].mean(), "mean_n_trades": group["n_trades"].mean(),
    })
summary = pd.DataFrame(rows).set_index("strategy")
print(summary.round(4))

## Block bootstrap: v2 vs v1, and v2 vs Whalley-Wilmott (the toughest opponent)

In [ ]:
def block_bootstrap_diff(df, strategy_a, strategy_b, col="terminal_pnl", n_boot=5000, seed=42):
    rng = np.random.default_rng(seed)
    unique_days = df["sample_date"].unique()
    a = df[df["strategy"] == strategy_a].set_index("sample_date")
    b = df[df["strategy"] == strategy_b].set_index("sample_date")
    observed_diff = a[col].mean() - b[col].mean()
    boot_diffs = np.zeros(n_boot)
    for i in range(n_boot):
        sampled_days = rng.choice(unique_days, size=len(unique_days), replace=True)
        a_vals = np.concatenate([a.loc[[d], col].values if d in a.index else [] for d in sampled_days])
        b_vals = np.concatenate([b.loc[[d], col].values if d in b.index else [] for d in sampled_days])
        boot_diffs[i] = a_vals.mean() - b_vals.mean()
    ci_lower, ci_upper = np.percentile(boot_diffs, [2.5, 97.5])
    p_value = 2 * min((boot_diffs < 0).mean(), (boot_diffs > 0).mean())
    return observed_diff, ci_lower, ci_upper, p_value

for opponent in ["deep_hedge", "whalley_wilmott", "bs_delta"]:
    print(f"=== deep_hedge_v2 vs {opponent} (P&L) ===")
    diff, lo, hi, p = block_bootstrap_diff(combined, "deep_hedge_v2", opponent, "terminal_pnl")
    print(f"  Mean diff: {diff:.4f}, 95% CI: [{lo:.4f}, {hi:.4f}], p={p:.4f}")
    diff_c, lo_c, hi_c, p_c = block_bootstrap_diff(combined, "deep_hedge_v2", opponent, "total_cost")
    print(f"  Mean cost diff: {diff_c:.4f}, 95% CI: [{lo_c:.4f}, {hi_c:.4f}], p={p_c:.4f}\n")